# Phase 0 — Notebook 00: Data Audit

## Hypothesis (this notebook tests no edge hypothesis)

This notebook does **not** test any hypothesis about a betting edge. Its purpose is to **characterize the data corpus** that all subsequent Phase 0 notebooks will rely on, and to **surface every quality issue** that will affect later modeling.

Specifically:

- Pull CFBD v2 metadata for FBS games across seasons 2015-2024 (regular + postseason).
- Count games, lines, opening-spread coverage, neutral-site games, FCS opponents, OT games, shortened/canceled games, duplicate IDs, and pre-game Elo coverage.
- For sample season 2024 only: pull plays and drives, snapshot real field schemas, and spot-check incomplete-PBP detection.
- Verify Open-Meteo Historical Weather API reachability and response shape.
- Produce per `(season, season_type)` coverage and exclusion tables.

**Exclusions documented here propagate to all downstream notebooks** via `research/data/data_quality_report.md`. If a `(season, season_type)` pair fails the Pre-game Elo coverage threshold (<80% per A.7), if a season's `spreadOpen` is too sparse (rule 16), or if play-by-play coverage is too thin (rule 20), Notebook 01 reads the report and excludes accordingly. The report is the single source of truth for "what's in the working set."

## Spec references

- `BUILD_SPEC.md` V5.1 rule 20 — data quality audit requirements
- `BUILD_SPEC.md` Owner addendum **A.7** (2026-05-07) — pre-game Elo replaces SP+/FPI; coverage threshold; `(season, season_type)` granularity
- `.cursorrules` rule 14 + Owner clarification 2026-05-07 — rating-timing lookahead trap
- `.cursorrules` rule 16 — market movement requires both opening and closing spreads
- `.cursorrules` rule 22 — STOP at end of Notebook 00; do not start Notebook 01 without approval

## Deliverables produced by this notebook

1. `research/data/cache/` — raw JSON cache (gitignored)
2. `research/data/cache/cfbd_call_log.csv` — per-call latency and budget log
3. `research/data/data_quality_report.md` — narrative report covering V5.1 rule 20 and A.7
4. `research/results/audit_summary.csv` — machine-readable counts per `(season, season_type)`
5. `research/results/audit_field_coverage.csv` — per-field non-null pct per `(season, season_type)`
6. `research/results/audit_sample_schema.json` — sample play and drive records from 2024

## What this notebook DOES NOT do

- Build `trigger_events` table — that's Notebook 01.
- Compute features — that's Notebooks 02a-g.
- Train any model — that's Notebook 03.
- Make any betting decisions — that's later.
- Pull plays or drives for any season except 2024 — Notebook 01 does the full backfill.
- Pull weather for any specific game — only one Open-Meteo reachability check.

## Call budget

CFBD v2 free tier = 1000 calls / month. This notebook is budgeted for **~89 CFBD calls + 1 Open-Meteo call**. Cache hits do not consume budget; only fresh network fetches do. The final cell prints the actual fresh budget consumed against the 1000-call cap.

In [ ]:
"""
Notebook 00 — imports, environment, path constants, fail-fast checks.

Run this cell first. If it raises, fix the error before continuing — none of
the downstream cells will work without these prerequisites.
"""
from __future__ import annotations

import csv
import hashlib
import json
import os
import pathlib
import time
from typing import Any

import httpx
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# --- Paths -------------------------------------------------------------------
NOTEBOOK_DIR = pathlib.Path(".").resolve()
RESEARCH_DIR = (NOTEBOOK_DIR / "..").resolve()
DATA_DIR = (RESEARCH_DIR / "data").resolve()
RESULTS_DIR = (RESEARCH_DIR / "results").resolve()
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CALL_LOG = CACHE_DIR / "cfbd_call_log.csv"
ENV_PATH = (RESEARCH_DIR / ".." / "backend" / ".env").resolve()

# --- Sanity check on workspace layout ----------------------------------------
assert RESEARCH_DIR.name == "research", (
    f"Expected to run inside research/notebooks/. Got NOTEBOOK_DIR={NOTEBOOK_DIR}. "
    f"cd into research/notebooks/ and re-launch jupyter."
)
assert ENV_PATH.exists(), (
    f"Did not find {ENV_PATH}. Per BUILD_SPEC.md A.5, the CFBD key lives in "
    f"backend/.env (NOT .env.example). Populate it before running this notebook."
)

# --- Load CFBD_API_KEY from backend/.env -------------------------------------
load_dotenv(ENV_PATH)

# --- FAIL FAST: refuse to run a single network call without the key ---------
assert os.environ.get("CFBD_API_KEY"), (
    "CFBD_API_KEY is not set. Expected to be populated in\n"
    f"    {ENV_PATH}\n"
    "Get a free key at https://collegefootballdata.com/key, then add the line\n"
    "    CFBD_API_KEY=your_key_here\n"
    "to backend/.env and re-run this cell. Do not run the rest of the notebook "
    "until this assertion passes — calls 30 of 89 failing on 401 wastes the "
    "entire budget for nothing."
)

print(f"[ok] paths resolved relative to {NOTEBOOK_DIR}")
print(f"[ok] CFBD_API_KEY loaded from {ENV_PATH}")
print(f"[ok] cache dir: {CACHE_DIR}")

In [ ]:
"""
HTTP helpers with disk cache and per-call logging.

cfbd_get(endpoint, **params) -> JSON
om_get(**params) -> JSON

Cache key = sha1 of sorted JSON of params, truncated to 16 chars.
Cache hit returns from disk; never touches the network and does not consume
budget. Cache miss writes the raw JSON to disk before returning.

Every call (hit or miss, success or failure) is logged to
cache/cfbd_call_log.csv with timestamp, service, endpoint, params hash,
cached flag, status, response size, elapsed ms.
"""
CFBD_BASE = "https://apinext.collegefootballdata.com"
OPEN_METEO_BASE = "https://archive-api.open-meteo.com/v1/archive"

if not CALL_LOG.exists():
    with CALL_LOG.open("w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(
            ["timestamp", "service", "endpoint", "params_hash", "cached",
             "status", "bytes", "elapsed_ms"]
        )


def _params_hash(params: dict) -> str:
    return hashlib.sha1(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]


def _cache_key(prefix: str, params: dict) -> pathlib.Path:
    return CACHE_DIR / f"{prefix}__{_params_hash(params)}.json"


def _log(service: str, endpoint: str, params: dict, *, cached: bool,
         status: int, bytes_: int, elapsed_ms: int) -> None:
    with CALL_LOG.open("a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(
            [time.strftime("%Y-%m-%dT%H:%M:%S"), service, endpoint,
             _params_hash(params), int(cached), status, bytes_, elapsed_ms]
        )


def cfbd_get(endpoint: str, force_refresh: bool = False, **params: Any) -> Any:
    key = _cache_key(f"cfbd__{endpoint.strip('/').replace('/', '_')}", params)
    if key.exists() and not force_refresh:
        size = key.stat().st_size
        data = json.loads(key.read_text(encoding="utf-8"))
        _log("cfbd", endpoint, params, cached=True, status=200,
             bytes_=size, elapsed_ms=0)
        return data
    headers = {
        "Authorization": f"Bearer {os.environ['CFBD_API_KEY']}",
        "Accept": "application/json",
    }
    t0 = time.perf_counter()
    r = httpx.get(f"{CFBD_BASE}{endpoint}", params=params,
                  headers=headers, timeout=120)
    elapsed_ms = int((time.perf_counter() - t0) * 1000)
    # Log BEFORE raise_for_status so failed calls still appear in the budget log.
    _log("cfbd", endpoint, params, cached=False, status=r.status_code,
         bytes_=len(r.content), elapsed_ms=elapsed_ms)
    r.raise_for_status()
    data = r.json()
    key.write_text(json.dumps(data), encoding="utf-8")
    return data


def om_get(force_refresh: bool = False, **params: Any) -> Any:
    key = _cache_key("openmeteo__archive", params)
    if key.exists() and not force_refresh:
        size = key.stat().st_size
        data = json.loads(key.read_text(encoding="utf-8"))
        _log("open_meteo", "/v1/archive", params, cached=True, status=200,
             bytes_=size, elapsed_ms=0)
        return data
    t0 = time.perf_counter()
    r = httpx.get(OPEN_METEO_BASE, params=params, timeout=60)
    elapsed_ms = int((time.perf_counter() - t0) * 1000)
    _log("open_meteo", "/v1/archive", params, cached=False, status=r.status_code,
         bytes_=len(r.content), elapsed_ms=elapsed_ms)
    r.raise_for_status()
    data = r.json()
    key.write_text(json.dumps(data), encoding="utf-8")
    return data


print("[ok] cfbd_get and om_get defined")
print(f"[ok] call log: {CALL_LOG}")

## Configuration

`SEASONS` covers 2015-2024 per the decision to probe the older seasons. `SEASON_TYPES` is regular + postseason; the audit reports each pair independently per A.7. `SAMPLE_SEASON` is the year used for plays/drives schema verification (2024 chosen for closest match to the swagger we just verified).

In [ ]:
SEASONS: list[int] = list(range(2015, 2025))   # 2015..2024 inclusive (10 seasons)
SEASON_TYPES: list[str] = ["regular", "postseason"]
SAMPLE_SEASON: int = 2024

# Sample weather coordinates: Bryant-Denny Stadium (Tuscaloosa), 2024-09-07 (Western Kentucky vs Alabama).
SAMPLE_VENUE_LAT: float = 33.21
SAMPLE_VENUE_LON: float = -87.55
SAMPLE_WEATHER_DATE: str = "2024-09-07"

# A.7 threshold for pre-game Elo coverage.
ELO_THRESHOLD_PCT: float = 80.0

# Plays-per-game threshold for incomplete PBP detection (V5.1 rule 20).
INCOMPLETE_PBP_THRESHOLD: int = 100

# Weeks 1..16 covers the FBS regular season (incl. conference championships).
WEEKS_REGULAR_FBS: list[int] = list(range(1, 17))

print(f"seasons: {SEASONS} ({len(SEASONS)} total)")
print(f"season types: {SEASON_TYPES}")
print(f"sample season for plays/drives: {SAMPLE_SEASON}")
print(f"elo coverage threshold: {ELO_THRESHOLD_PCT}%")

## Open-Meteo reachability

Single call against Bryant-Denny coordinates. Verifies the Historical Weather API endpoint, units (Fahrenheit / mph / inch), and the hourly variable list (`temperature_2m`, `wind_speed_10m`, `precipitation`, `weather_code`). One sample is sufficient — full per-game weather is Notebook 01's concern, and venue geocoding is its own separate problem.

In [ ]:
om_response = om_get(
    latitude=SAMPLE_VENUE_LAT,
    longitude=SAMPLE_VENUE_LON,
    start_date=SAMPLE_WEATHER_DATE,
    end_date=SAMPLE_WEATHER_DATE,
    hourly="temperature_2m,wind_speed_10m,precipitation,weather_code",
    temperature_unit="fahrenheit",
    wind_speed_unit="mph",
    precipitation_unit="inch",
)

om_keys = sorted(om_response.keys())
om_hourly_keys = sorted(om_response.get("hourly", {}).keys())
om_n_hourly = len(om_response.get("hourly", {}).get("time", []))
om_units = om_response.get("hourly_units", {})

print(f"top-level keys: {om_keys}")
print(f"hourly keys: {om_hourly_keys}")
print(f"hourly samples returned: {om_n_hourly}")
print(
    f"units: temp={om_units.get('temperature_2m')}, "
    f"wind={om_units.get('wind_speed_10m')}, "
    f"precip={om_units.get('precipitation')}"
)
om_reachable = (
    om_n_hourly > 0
    and {"temperature_2m", "wind_speed_10m", "precipitation", "weather_code"}.issubset(set(om_hourly_keys))
)
print(f"[{'ok' if om_reachable else 'FAIL'}] open-meteo reachability")

## CFBD metadata pulls

Pull `/teams/fbs`, `/games`, `/lines` for every `(season, season_type)` pair, plus `/ratings/sp` and `/ratings/fpi` (one call per season — these are stored as evidence for the A.7 rejection rationale, not used as features).

Budget for this section: 10 (teams) + 20 (games) + 20 (lines) + 10 (sp) + ~10 (fpi, may 404 on older seasons) = **~70 CFBD calls**. Cached after first run.

In [ ]:
# /teams/fbs — one call per season (10 calls).
fbs_team_counts: dict[int, int] = {}
for year in SEASONS:
    teams = cfbd_get("/teams/fbs", year=year)
    fbs_team_counts[year] = len(teams)

print("FBS team counts per season:")
for year, n in fbs_team_counts.items():
    print(f"  {year}: {n}")

In [ ]:
# /games — one call per (season, season_type) pair (20 calls).
games_records: list[dict] = []
for year in SEASONS:
    for season_type in SEASON_TYPES:
        games = cfbd_get(
            "/games",
            year=year,
            seasonType=season_type,
            classification="fbs",
        )
        for g in games:
            g["_audit_season"] = year
            g["_audit_season_type"] = season_type
            games_records.append(g)

games_df = pd.json_normalize(games_records)
print(f"games rows: {len(games_df)}")
print(f"games columns ({len(games_df.columns)}):")
for c in sorted(games_df.columns.tolist()):
    print(f"  {c}")

In [ ]:
# /lines — one call per (season, season_type) pair (20 calls).
# Response is one entry per game; each entry has a `lines` array of provider
# rows. We retain the nested structure for the provider-availability matrix.
lines_records: list[dict] = []
for year in SEASONS:
    for season_type in SEASON_TYPES:
        lines = cfbd_get(
            "/lines",
            year=year,
            seasonType=season_type,
            classification="fbs",
        )
        for entry in lines:
            entry["_audit_season"] = year
            entry["_audit_season_type"] = season_type
            lines_records.append(entry)

print(f"lines entries (one per game): {len(lines_records)}")
if lines_records:
    sample = lines_records[0]
    sample_top = sorted(sample.keys())
    sample_book = sample.get("lines") or []
    sample_book_keys = sorted(sample_book[0].keys()) if sample_book else []
    print(f"sample top-level keys: {sample_top}")
    print(f"sample provider entry keys: {sample_book_keys}")

In [ ]:
# /ratings/sp and /ratings/fpi — evidence for A.7 rejection rationale.
# Pulled here NOT for use as features (rule 14 lookahead leak) but to confirm
# in code that the response shape has no `week` field, which is the evidence
# A.7 documents in narrative form.
ratings_sp_records: list[dict] = []
ratings_fpi_records: list[dict] = []
ratings_fpi_404_seasons: list[int] = []

for year in SEASONS:
    sp = cfbd_get("/ratings/sp", year=year)
    for r in sp:
        r["_audit_season"] = year
        ratings_sp_records.append(r)
    try:
        fpi = cfbd_get("/ratings/fpi", year=year)
    except httpx.HTTPStatusError as exc:
        if exc.response.status_code in (404, 400):
            ratings_fpi_404_seasons.append(year)
            print(f"[note] /ratings/fpi year={year} returned {exc.response.status_code}")
            continue
        raise
    else:
        for r in fpi:
            r["_audit_season"] = year
            ratings_fpi_records.append(r)

print(f"sp rating rows total: {len(ratings_sp_records)}")
print(f"fpi rating rows total: {len(ratings_fpi_records)}")
print(f"fpi seasons unavailable: {ratings_fpi_404_seasons}")

# Confirm absence of any week-like field — direct evidence for A.7.
if ratings_sp_records:
    sp_keys = sorted(ratings_sp_records[0].keys())
    sp_has_week = any("week" in k.lower() for k in sp_keys)
    print(f"[A.7 evidence] /ratings/sp first-row keys: {sp_keys}")
    print(f"[A.7 evidence] /ratings/sp has any 'week' field: {sp_has_week}")
if ratings_fpi_records:
    fpi_keys = sorted(ratings_fpi_records[0].keys())
    fpi_has_week = any("week" in k.lower() for k in fpi_keys)
    print(f"[A.7 evidence] /ratings/fpi first-row keys: {fpi_keys}")
    print(f"[A.7 evidence] /ratings/fpi has any 'week' field: {fpi_has_week}")

## Sample-season schema verification (2024 only)

Pull plays and drives for the sample season. Capture one example play and one example drive into `audit_sample_schema.json` for Notebook 01 to assert against later. Also reuses the plays for the per-game incomplete-PBP check.

Budget for this section: ~16 plays calls (one per regular-season week) + 1 drives call = **~17 CFBD calls**. Cached after first run.

In [ ]:
# /plays for SAMPLE_SEASON regular-season weeks. Iterate weeks 1..16.
plays_records: list[dict] = []
plays_failed_weeks: list[int] = []
for week in WEEKS_REGULAR_FBS:
    try:
        plays = cfbd_get(
            "/plays",
            year=SAMPLE_SEASON,
            week=week,
            classification="fbs",
            seasonType="regular",
        )
    except httpx.HTTPStatusError as exc:
        plays_failed_weeks.append(week)
        print(f"[note] /plays year={SAMPLE_SEASON} week={week} returned {exc.response.status_code}")
        continue
    for p in plays:
        p["_audit_week"] = week
        plays_records.append(p)

print(f"plays for {SAMPLE_SEASON} regular: {len(plays_records)} rows across {len(WEEKS_REGULAR_FBS) - len(plays_failed_weeks)} weeks")
if plays_failed_weeks:
    print(f"failed weeks: {plays_failed_weeks}")

In [ ]:
# /drives for SAMPLE_SEASON regular season — single call.
drives_records = cfbd_get("/drives", year=SAMPLE_SEASON, seasonType="regular", classification="fbs")
print(f"drives for {SAMPLE_SEASON} regular: {len(drives_records)} rows")
if drives_records:
    print(f"drive first-row keys: {sorted(drives_records[0].keys())}")

In [ ]:
# Save play and drive sample schemas to audit_sample_schema.json (deliverable 6).
sample_schema = {
    "captured_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "sample_season": SAMPLE_SEASON,
    "play_example": plays_records[0] if plays_records else None,
    "play_field_names": sorted(plays_records[0].keys()) if plays_records else [],
    "drive_example": drives_records[0] if drives_records else None,
    "drive_field_names": sorted(drives_records[0].keys()) if drives_records else [],
}

schema_path = RESULTS_DIR / "audit_sample_schema.json"
schema_path.write_text(json.dumps(sample_schema, indent=2, default=str), encoding="utf-8")
print(f"wrote {schema_path}")
print(f"play fields ({len(sample_schema['play_field_names'])}): {sample_schema['play_field_names']}")
print(f"drive fields ({len(sample_schema['drive_field_names'])}): {sample_schema['drive_field_names']}")

## Per-game flagging (V5.1 rule 20 + A.7)

Build derived columns for every game across every `(season, season_type)` pair: classification flags, pre-game Elo presence, OT detection via line-score length, shortened/canceled detection, line-coverage joins. Each derived flag corresponds to a specific exclusion rule.

In [ ]:
def _classify(value: Any) -> str:
    """CFBD DivisionClassification can be a bare string or a wrapped object; normalize."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value.lower()
    if isinstance(value, dict):
        return str(value.get("name") or value.get("value") or "").lower()
    return str(value).lower()


def _line_scores_len(value: Any) -> int | None:
    if isinstance(value, list):
        return len(value)
    return None


# Classification flags ------------------------------------------------------
games_df["home_class"] = games_df["homeClassification"].apply(_classify)
games_df["away_class"] = games_df["awayClassification"].apply(_classify)
games_df["fbs_vs_fbs"] = (games_df["home_class"] == "fbs") & (games_df["away_class"] == "fbs")
games_df["fbs_vs_fcs"] = (
    ((games_df["home_class"] == "fbs") & (games_df["away_class"] == "fcs"))
    | ((games_df["home_class"] == "fcs") & (games_df["away_class"] == "fbs"))
)

# Pre-game Elo presence (A.7) ----------------------------------------------
games_df["has_pregame_elo_both"] = (
    games_df["homePregameElo"].notna() & games_df["awayPregameElo"].notna()
)

# Line-score length-based flags --------------------------------------------
games_df["home_ls_len"] = games_df["homeLineScores"].apply(_line_scores_len)
games_df["away_ls_len"] = games_df["awayLineScores"].apply(_line_scores_len)
_completed_bool = games_df["completed"].fillna(False).astype(bool)
games_df["likely_ot"] = _completed_bool & games_df["home_ls_len"].fillna(0).astype(int).gt(4)
games_df["shortened_or_canceled"] = (
    (~_completed_bool)
    | (_completed_bool & games_df["home_ls_len"].fillna(0).astype(int).lt(4))
)

# Sanity prints -------------------------------------------------------------
print(f"games_df rows: {len(games_df)}")
print(f"completed: {int(_completed_bool.sum())}")
print(f"fbs_vs_fbs: {int(games_df['fbs_vs_fbs'].sum())}")
print(f"fbs_vs_fcs: {int(games_df['fbs_vs_fcs'].sum())}")
print(f"has_pregame_elo_both: {int(games_df['has_pregame_elo_both'].sum())}")
print(f"likely_ot: {int(games_df['likely_ot'].sum())}")
print(f"shortened_or_canceled: {int(games_df['shortened_or_canceled'].sum())}")

In [ ]:
# Build a lines-summary keyed by gameId, then merge into games_df.
lines_summary_rows: list[dict] = []
for entry in lines_records:
    book_lines = entry.get("lines") or []
    n_providers = len(book_lines)
    spreads = [b.get("spread") for b in book_lines if b.get("spread") is not None]
    spreads_open = [b.get("spreadOpen") for b in book_lines if b.get("spreadOpen") is not None]
    has_any_spread = len(spreads) > 0
    has_any_open = len(spreads_open) > 0
    has_open_and_close = any(
        (b.get("spread") is not None and b.get("spreadOpen") is not None)
        for b in book_lines
    )
    providers = sorted({(b.get("provider") or "").strip() for b in book_lines if b.get("provider")})
    lines_summary_rows.append({
        "id": entry.get("id"),
        "n_providers": n_providers,
        "has_any_spread": has_any_spread,
        "has_any_opening_spread": has_any_open,
        "has_open_and_close": has_open_and_close,
    })
lines_summary_df = pd.DataFrame(lines_summary_rows)
print(f"lines_summary_df rows: {len(lines_summary_df)}")

games_df = games_df.merge(lines_summary_df, on="id", how="left")
games_df["n_providers"] = games_df["n_providers"].fillna(0).astype(int)
games_df["has_any_spread"] = games_df["has_any_spread"].fillna(False).astype(bool)
games_df["has_any_opening_spread"] = games_df["has_any_opening_spread"].fillna(False).astype(bool)
games_df["has_open_and_close"] = games_df["has_open_and_close"].fillna(False).astype(bool)
games_df["has_lines_entry"] = games_df["n_providers"] > 0

print(f"games with any spread: {int(games_df['has_any_spread'].sum())}")
print(f"games with any opening spread: {int(games_df['has_any_opening_spread'].sum())}")
print(f"games with both open and close: {int(games_df['has_open_and_close'].sum())}")

In [ ]:
# Final per-game exclusion flag and per-reason breakdowns.
_completed_bool = games_df["completed"].fillna(False).astype(bool)

games_df["passes_exclusion_filter"] = (
    _completed_bool
    & games_df["fbs_vs_fbs"]
    & games_df["has_any_spread"]
    & ~games_df["shortened_or_canceled"]
)

# Per-reason exclusion (a single game can fail multiple reasons).
games_df["excluded_for_not_completed"] = ~_completed_bool
games_df["excluded_for_fcs_opponent"] = ~games_df["fbs_vs_fbs"]
games_df["excluded_for_no_spread"] = ~games_df["has_any_spread"]
games_df["excluded_for_canceled_or_shortened"] = games_df["shortened_or_canceled"]

print(f"games passing exclusion filter: {int(games_df['passes_exclusion_filter'].sum())}")

In [ ]:
# Classification schema-drift detection (V5.1 rule 20).
#
# _classify() above defensively normalizes unrecognized homeClassification /
# awayClassification shapes to the empty string so the per-game flags don't
# crash. A silent fallback would, however, mis-label drifted rows as
# not-FBS-vs-FBS and quietly remove them from the working set. Detect any
# such rows here and bubble the count into data_quality_report.md so the
# drift is impossible to miss.
def _classification_unrecognized(value: Any) -> bool:
    if value is None:
        return False  # plain missing; not a drift signal
    if isinstance(value, str):
        return False
    if isinstance(value, dict) and ("name" in value or "value" in value):
        return False
    return True


games_df["home_class_unrecognized"] = games_df["homeClassification"].apply(
    _classification_unrecognized
)
games_df["away_class_unrecognized"] = games_df["awayClassification"].apply(
    _classification_unrecognized
)
games_df["classification_unrecognized"] = (
    games_df["home_class_unrecognized"] | games_df["away_class_unrecognized"]
)

classification_drift_total = int(games_df["classification_unrecognized"].sum())
if classification_drift_total == 0:
    classification_drift_df = pd.DataFrame(
        columns=["season", "season_type", "n_unrecognized"]
    )
    print("[ok] no classification schema drift detected")
else:
    classification_drift_df = (
        games_df[games_df["classification_unrecognized"]]
        .groupby(["_audit_season", "_audit_season_type"], dropna=False)
        .size()
        .reset_index(name="n_unrecognized")
        .rename(columns={"_audit_season": "season", "_audit_season_type": "season_type"})
        .sort_values(["season", "season_type"])
        .reset_index(drop=True)
    )
    print(
        f"[warn] classification schema drift detected on "
        f"{classification_drift_total} game(s):"
    )
    print(classification_drift_df.to_string(index=False))

## Per `(season, season_type)` summary — `audit_summary.csv`

GroupBy aggregation. One row per `(season, season_type)`. All count columns required by V5.1 rule 20, plus per-exclusion-reason breakdowns and the A.7 Elo coverage column with the 80% threshold flag.

In [ ]:
audit_summary_rows: list[dict] = []
for (season, season_type), group in games_df.groupby(
    ["_audit_season", "_audit_season_type"], dropna=False
):
    n_total = len(group)
    n_fbs_vs_fbs = int(group["fbs_vs_fbs"].sum())
    n_fbs_with_elo = int((group["fbs_vs_fbs"] & group["has_pregame_elo_both"]).sum())
    pct_elo_coverage = (
        100.0 * n_fbs_with_elo / n_fbs_vs_fbs if n_fbs_vs_fbs > 0 else float("nan")
    )
    passes_elo = (pct_elo_coverage >= ELO_THRESHOLD_PCT) if n_fbs_vs_fbs > 0 else False
    audit_summary_rows.append({
        "season": int(season),
        "season_type": str(season_type),
        "n_games_total": n_total,
        "n_games_completed": int(group["completed"].fillna(False).astype(bool).sum()),
        "n_games_fbs_vs_fbs": n_fbs_vs_fbs,
        "n_games_fbs_vs_fcs": int(group["fbs_vs_fcs"].sum()),
        "n_games_neutral_site": int(group["neutralSite"].fillna(False).astype(bool).sum()),
        "n_games_with_pregame_elo_both": int(group["has_pregame_elo_both"].sum()),
        "n_games_fbs_with_pregame_elo_both": n_fbs_with_elo,
        "pct_pregame_elo_coverage_fbs_vs_fbs": (
            round(pct_elo_coverage, 2) if pd.notna(pct_elo_coverage) else float("nan")
        ),
        "passes_elo_threshold_80pct": passes_elo,
        "n_games_with_lines": int(group["has_lines_entry"].sum()),
        "n_games_with_any_spread": int(group["has_any_spread"].sum()),
        "n_games_with_any_opening_spread": int(group["has_any_opening_spread"].sum()),
        "n_games_with_opening_and_closing": int(group["has_open_and_close"].sum()),
        "n_games_likely_ot": int(group["likely_ot"].sum()),
        "n_games_shortened_or_canceled": int(group["shortened_or_canceled"].sum()),
        "n_games_dup_id": int(group["id"].duplicated().sum()),
        "excluded_for_not_completed": int(group["excluded_for_not_completed"].sum()),
        "excluded_for_fcs_opponent": int(group["excluded_for_fcs_opponent"].sum()),
        "excluded_for_no_spread": int(group["excluded_for_no_spread"].sum()),
        "excluded_for_canceled_or_shortened": int(group["excluded_for_canceled_or_shortened"].sum()),
        "n_games_after_exclusions": int(group["passes_exclusion_filter"].sum()),
    })

audit_summary_df = (
    pd.DataFrame(audit_summary_rows)
    .sort_values(["season", "season_type"])
    .reset_index(drop=True)
)

audit_summary_path = RESULTS_DIR / "audit_summary.csv"
audit_summary_df.to_csv(audit_summary_path, index=False)
print(f"wrote {audit_summary_path}")
print(audit_summary_df.to_string(index=False))

## Field coverage — `audit_field_coverage.csv`

For each games-table field, compute pct-non-null per `(season, season_type)`. Catches fields that vanish in older seasons (likely candidates: `homePregameElo`, `excitementIndex`, `attendance`).

In [ ]:
field_coverage_rows: list[dict] = []
for (season, season_type), group in games_df.groupby(
    ["_audit_season", "_audit_season_type"], dropna=False
):
    n = len(group)
    if n == 0:
        continue
    for col in group.columns:
        if col.startswith("_audit_"):
            continue
        non_null_pct = 100.0 * group[col].notna().sum() / n
        field_coverage_rows.append({
            "season": int(season),
            "season_type": str(season_type),
            "field": col,
            "n_games": n,
            "pct_non_null": round(non_null_pct, 2),
        })

field_coverage_df = (
    pd.DataFrame(field_coverage_rows)
    .sort_values(["season", "season_type", "field"])
    .reset_index(drop=True)
)

field_coverage_path = RESULTS_DIR / "audit_field_coverage.csv"
field_coverage_df.to_csv(field_coverage_path, index=False)
print(f"wrote {field_coverage_path}")
print(f"rows: {len(field_coverage_df)}")
print(f"unique fields covered: {field_coverage_df['field'].nunique()}")

## Pre-game Elo coverage by `(season, season_type)` — A.7

Per addendum A.7, every `(season, season_type)` pair must have ≥80% pre-game Elo coverage among FBS-vs-FBS games to be eligible for training. Pairs failing the threshold are excluded — regular and postseason are evaluated independently.

In [ ]:
elo_coverage = (
    audit_summary_df[[
        "season",
        "season_type",
        "n_games_fbs_vs_fbs",
        "n_games_fbs_with_pregame_elo_both",
        "pct_pregame_elo_coverage_fbs_vs_fbs",
        "passes_elo_threshold_80pct",
    ]]
    .copy()
    .sort_values(["season", "season_type"])
    .reset_index(drop=True)
)

elo_exclusions = elo_coverage[~elo_coverage["passes_elo_threshold_80pct"].astype(bool)].copy()

print("== Pre-game Elo coverage by (season, season_type) ==")
print(elo_coverage.to_string(index=False))
print()
print(f"== (season, season_type) pairs failing the {ELO_THRESHOLD_PCT}% threshold ==")
if elo_exclusions.empty:
    print("[ok] all (season, season_type) pairs pass the threshold")
else:
    print(elo_exclusions.to_string(index=False))

## /lines provider availability matrix

Which sportsbooks (Bovada, DraftKings, ESPN Bet, etc.) appear in CFBD's `/lines` data per `(season, season_type)`? Provider mix has changed over time, and the audit needs to surface that for future provider-specific decisions.

In [ ]:
provider_rows: list[dict] = []
for entry in lines_records:
    season = entry["_audit_season"]
    st = entry["_audit_season_type"]
    for b in entry.get("lines") or []:
        prov = (b.get("provider") or "").strip()
        if prov:
            provider_rows.append({"season": season, "season_type": st, "provider": prov})

provider_df = pd.DataFrame(provider_rows)
if provider_df.empty:
    provider_matrix = pd.DataFrame()
    print("[note] no provider data in /lines responses")
else:
    provider_matrix = (
        provider_df.groupby(["season", "season_type", "provider"]).size()
        .unstack("provider", fill_value=0)
        .sort_index()
    )
    print("== Provider availability matrix (count of game-provider rows) ==")
    print(provider_matrix.to_string())

## Incomplete play-by-play spot check (sample season 2024)

Group plays by `gameId` for 2024 regular season. Flag games with fewer than `INCOMPLETE_PBP_THRESHOLD` plays (<100). This is a single-season sample — Notebook 01 will run the same check for the full corpus when it pulls all plays.

In [ ]:
plays_df = pd.json_normalize(plays_records) if plays_records else pd.DataFrame()
plays_count_per_game: pd.Series = pd.Series(dtype=int)
incomplete_pbp_n = 0
plays_pbp_count_col: str | None = None

if plays_df.empty:
    print("[skip] no plays loaded; nothing to check")
else:
    # CFBD play schema typically uses `gameId` (camelCase). Fall back to other plausible names.
    for candidate in ("gameId", "game_id", "gameID"):
        if candidate in plays_df.columns:
            plays_pbp_count_col = candidate
            break
    if plays_pbp_count_col is None:
        print(
            f"[note] /plays response shape unexpected; columns: "
            f"{sorted(plays_df.columns.tolist())[:30]}"
        )
    else:
        plays_count_per_game = plays_df.groupby(plays_pbp_count_col).size()
        incomplete_pbp_n = int((plays_count_per_game < INCOMPLETE_PBP_THRESHOLD).sum())
        print(f"games in {SAMPLE_SEASON} regular plays sample: {len(plays_count_per_game)}")
        print(f"games with <{INCOMPLETE_PBP_THRESHOLD} plays in sample: {incomplete_pbp_n}")
        print("plays-per-game distribution:")
        print(plays_count_per_game.describe().to_string())

## Write `data_quality_report.md`

Single output file aggregating every audit dimension above. Per A.3 + A.5, this lives at `research/data/data_quality_report.md` (gitignored — it's a regenerated local audit artifact).

In [ ]:
# Build data_quality_report.md (deliverable 3).
report_path = DATA_DIR / "data_quality_report.md"


def _df_block(df: pd.DataFrame) -> str:
    """Render a DataFrame as a fenced text block (avoids tabulate dep)."""
    if df is None or df.empty:
        return "*(empty)*\n"
    return "```\n" + df.to_string(index=False) + "\n```\n"


def _series_block(s: pd.Series) -> str:
    if s is None or s.empty:
        return "*(empty)*\n"
    return "```\n" + s.to_string() + "\n```\n"


# --- Recompute call totals from the on-disk log so the report reflects truth.
with CALL_LOG.open(encoding="utf-8") as _f:
    _log_rows = list(csv.DictReader(_f))
n_cfbd_fresh = sum(1 for r in _log_rows if r["service"] == "cfbd" and r["cached"] == "0")
n_cfbd_cached = sum(1 for r in _log_rows if r["service"] == "cfbd" and r["cached"] == "1")
n_om_fresh = sum(1 for r in _log_rows if r["service"] == "open_meteo" and r["cached"] == "0")
n_om_cached = sum(1 for r in _log_rows if r["service"] == "open_meteo" and r["cached"] == "1")

# --- Per-pair verdict lines ------------------------------------------------
verdict_lines: list[str] = []
for _, row in audit_summary_df.iterrows():
    s_, st_ = int(row["season"]), str(row["season_type"])
    elo_ok = bool(row["passes_elo_threshold_80pct"])
    pct_elo = row["pct_pregame_elo_coverage_fbs_vs_fbs"]
    pct_elo_str = f"{pct_elo:.1f}%" if pd.notna(pct_elo) else "n/a"
    n_after = int(row["n_games_after_exclusions"])
    if not elo_ok:
        verdict_lines.append(
            f"- **{s_} {st_}**: EXCLUDE — Elo coverage {pct_elo_str} < {ELO_THRESHOLD_PCT:.0f}% (A.7 threshold)"
        )
    elif n_after < 30:
        verdict_lines.append(
            f"- **{s_} {st_}**: USABLE-BUT-THIN — {n_after} games post-exclusion (Elo coverage {pct_elo_str})"
        )
    else:
        verdict_lines.append(
            f"- **{s_} {st_}**: USABLE — {n_after} games post-exclusion (Elo coverage {pct_elo_str})"
        )
verdict_block = "\n".join(verdict_lines)

# --- Endpoint table --------------------------------------------------------
endpoints_block = (
    "| Service | Calls (fresh) | Calls (cached) | Notes |\n"
    "|---|---|---|---|\n"
    f"| CFBD v2 | {n_cfbd_fresh} | {n_cfbd_cached} | year-keyed metadata + sample {SAMPLE_SEASON} plays/drives |\n"
    f"| Open-Meteo | {n_om_fresh} | {n_om_cached} | reachability check only |\n"
    f"| **Total fresh CFBD** | **{n_cfbd_fresh}** | — | budget consumed this run |\n"
)

# --- A.7 evidence string ---------------------------------------------------
a7_evidence_lines: list[str] = []
if ratings_sp_records:
    sp_keys_first = sorted(ratings_sp_records[0].keys())
    sp_has_week = any("week" in k.lower() for k in sp_keys_first)
    a7_evidence_lines.append(
        f"- `/ratings/sp` first-row keys: `{sp_keys_first}` — has any 'week' field: **{sp_has_week}**"
    )
if ratings_fpi_records:
    fpi_keys_first = sorted(ratings_fpi_records[0].keys())
    fpi_has_week = any("week" in k.lower() for k in fpi_keys_first)
    a7_evidence_lines.append(
        f"- `/ratings/fpi` first-row keys: `{fpi_keys_first}` — has any 'week' field: **{fpi_has_week}**"
    )
if ratings_fpi_404_seasons:
    a7_evidence_lines.append(
        f"- `/ratings/fpi` returned 404/400 for seasons: `{ratings_fpi_404_seasons}` — coverage further limited"
    )
a7_evidence_block = "\n".join(a7_evidence_lines) if a7_evidence_lines else "*(no evidence captured)*"

# --- Plays distribution block ---------------------------------------------
plays_dist_block = (
    _series_block(plays_count_per_game.describe())
    if not plays_count_per_game.empty
    else "*(no plays loaded)*\n"
)

# --- Dynamic limitations (schema drift) ----------------------------------
if classification_drift_total > 0:
    pairs_str = "; ".join(
        f"({int(row.season)}, {row.season_type}): {int(row.n_unrecognized)}"
        for row in classification_drift_df.itertuples(index=False)
    )
    dynamic_limitations_block = (
        f"- **Classification schema drift detected.** "
        f"{classification_drift_total} game(s) had unrecognized "
        f"`homeClassification` or `awayClassification` shapes (neither "
        f"bare string nor object with `name`/`value`); these were "
        f"defensively normalized to empty string by `_classify`, which "
        f"silently treats them as not-FBS-vs-FBS. Affected "
        f"`(season, season_type)` pairs with game counts: "
        f"{pairs_str}. Notebook 01 should validate the actual "
        f"classification payload before relying on `fbs_vs_fbs`.\n"
    )
else:
    dynamic_limitations_block = ""

# --- Build the report -----------------------------------------------------
provider_block = (
    _df_block(provider_matrix.reset_index()) if not provider_matrix.empty else "*(no provider data)*\n"
)

report = f"""# Data Quality Report — Notebook 00

**Generated:** {time.strftime("%Y-%m-%d %H:%M:%S")}
**Spec:** `BUILD_SPEC.md` V5.1 rule 20 + Owner addendum **A.7** (2026-05-07)
**Seasons audited:** {SEASONS[0]}-{SEASONS[-1]} ({len(SEASONS)} seasons)
**Season types:** {SEASON_TYPES}
**Sample season for plays/drives:** {SAMPLE_SEASON}

This report is a regenerated local audit artifact — it lives in the gitignored
`research/data/` directory and is rebuilt every time Notebook 00 is rerun. It
is the single source of truth for `(season, season_type)` exclusions that all
downstream notebooks must respect.

## Endpoints exercised

{endpoints_block}
CFBD v2 free tier = 1000 calls/month. Per-call detail tracked in `research/data/cache/cfbd_call_log.csv`.

## Per `(season, season_type)` summary

{_df_block(audit_summary_df)}

## Exclusion rules applied

A game is excluded from the working set used by Notebook 01 if **any** of the following holds:

- `not completed` OR (completed AND `len(homeLineScores) < 4`) — canceled or shortened
- Either team is not FBS — FCS opponent
- No `/lines` entry has a non-null `spread` — missing spread

`(season, season_type)`-level exclusions added by addendum **A.7**:

- Pre-game Elo coverage among FBS-vs-FBS games < {ELO_THRESHOLD_PCT:.0f}% — pair excluded entirely from Elo-feature training (regular and postseason evaluated independently).

OT handling is enforced at the trigger level, not the game level, per addendum **A.2** — overtime triggers are excluded at ingest in Notebook 01; regulation triggers from the same game still contribute.

## Pre-game Elo coverage by `(season, season_type)`

Per addendum A.7. Coverage = (FBS-vs-FBS games with both `homePregameElo` and `awayPregameElo` non-null) / (FBS-vs-FBS games), as a percent.

{_df_block(elo_coverage)}

`(season, season_type)` pairs failing the {ELO_THRESHOLD_PCT:.0f}% threshold:

{_df_block(elo_exclusions)}

## A.7 evidence — why not SP+/FPI

Direct response inspection from this run (data, not narrative):

{a7_evidence_block}

`/ratings/sp` and `/ratings/fpi` both return season-end rating values for any past season; neither response carries a week dimension. Using either as a feature for an in-season trigger is a rule-3 (R3) lookahead leak. SP+/FPI are added to `validated_filters.json.rejected_features` at Notebook 03 close.

## /lines provider availability matrix

Count of game-provider rows per `(season, season_type)`. Useful when deciding which providers' lines to trust at scale.

{provider_block}

## Sample-season schema verification ({SAMPLE_SEASON})

- `/plays` returned **{len(plays_records)}** rows across **{len(WEEKS_REGULAR_FBS) - len(plays_failed_weeks)} of {len(WEEKS_REGULAR_FBS)}** regular-season weeks
- `/drives` returned **{len(drives_records)}** rows
- Sample play field count: **{len(sample_schema['play_field_names'])}**
- Sample drive field count: **{len(sample_schema['drive_field_names'])}**
- Schemas saved to: `research/results/audit_sample_schema.json`

Plays-per-game distribution (sample season {SAMPLE_SEASON}):

{plays_dist_block}

Games with <{INCOMPLETE_PBP_THRESHOLD} plays in sample: **{incomplete_pbp_n}**. Notebook 01 must repeat this check for all {len(SEASONS)} seasons when it pulls the full plays corpus, and add affected games to its own per-game exclusion list.

## Open-Meteo reachability

- Endpoint: `{OPEN_METEO_BASE}`
- Sample location: Bryant-Denny Stadium ({SAMPLE_VENUE_LAT}, {SAMPLE_VENUE_LON}), {SAMPLE_WEATHER_DATE}
- Top-level keys returned: `{om_keys}`
- Hourly variable keys returned: `{om_hourly_keys}`
- Hourly samples returned: **{om_n_hourly}**
- Units (verified): temperature_2m=`{om_units.get('temperature_2m')}`, wind_speed_10m=`{om_units.get('wind_speed_10m')}`, precipitation=`{om_units.get('precipitation')}`

Verdict: **{'reachable' if om_reachable else 'NOT reachable'}**. Variables and units {'match expected' if om_reachable else 'do NOT match expected — investigate before Notebook 01'}. Notebook 01 venue geocoding is the next concern; per-game weather pulls are not in scope here.

## Limitations encountered
{dynamic_limitations_block}
- **CFBD v2 has no weekly SP+/FPI endpoint.** `/ratings/sp` and `/ratings/fpi` accept year (and team) only; both return season-end values for any past season. Per A.7, pre-game Elo from `/games` is the substitute. SP+/FPI go to `validated_filters.json.rejected_features` at Notebook 03.
- **Per-game play counts only computed for sample season {SAMPLE_SEASON}.** Notebook 01 will run the full check across {SEASONS[0]}-{SEASONS[-1]} when it pulls the full plays corpus.
- **Open-Meteo coverage at scale is not verified.** Only one sample call. Venue geocoding and per-game weather pulls are Notebook 01 work.
- **Cache invalidation is manual.** If CFBD updates a season's data later, the on-disk cache holds stale values until `force_refresh=True` is used. Acceptable for backtest; documented per addendum A.5.
- **CFBD v2 free-tier monthly cap = 1000 calls.** Notebook 01 will need ~256 calls just for plays. Plan to chunk across two billing cycles OR upgrade to the $10/mo Patreon Tier 3 before running Notebook 01.
- **The Odds API is not exercised in this notebook.** Phase 0 historical-line audit relies on CFBD `/lines`. The Odds API only enters Phase 4 (live snapshots).

## Verdict — per-pair usability for Notebook 01

{verdict_block}

Pairs labeled **USABLE** are eligible to feed Notebook 01. Pairs labeled **EXCLUDE** are dropped per A.7. Pairs labeled **USABLE-BUT-THIN** have <30 games post-exclusion and may not produce statistically meaningful trigger counts in any (deficit, quarter) bucket — this surfaces in Notebook 03's bucket-count check (Phase 0 acceptance gate 4).

End of report.
"""

report_path.write_text(report, encoding="utf-8")
print(f"wrote {report_path} ({len(report):,} chars)")

## Call budget

Final tally. Reads `cfbd_call_log.csv` and reports fresh CFBD calls consumed against the 1000/month free-tier cap.

In [ ]:
with CALL_LOG.open(encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

n_fresh_cfbd = sum(1 for r in rows if r["service"] == "cfbd" and r["cached"] == "0")
n_cached_cfbd = sum(1 for r in rows if r["service"] == "cfbd" and r["cached"] == "1")
n_fresh_om = sum(1 for r in rows if r["service"] == "open_meteo" and r["cached"] == "0")

print(f"Used {n_fresh_cfbd} / 1000 free-tier calls this run.")
print(
    f"(plus {n_cached_cfbd} CFBD cache hits and {n_fresh_om} Open-Meteo fresh calls, "
    f"neither of which counts against the cap)"
)